In [ ]:
import os
import json
import time
import pandas as pd
from PIL import Image
import google.generativeai as genai

# SETUP & CONFIGURATION
API_KEY = "xxxxxxxxxxxxxxxxxxxx" # Replace with your actual Gemini API key
genai.configure(api_key=API_KEY)

# Using gemini-2.5-flash for rapid, cost-effective vision processing
model = genai.GenerativeModel(
    model_name="gemini-2.5-flash",
    generation_config={"response_mime_type": "application/json"}
)

# PROMPT ENGINEERING
SYSTEM_PROMPT = """
You are an expert socio-economic surveyor assessing homes. 
Look closely at the provided photo and extract its structural and interior features. 

CRITICAL RULE FOR COUNTING STORIES:
Indian houses often have a small staircase head-room (mummy/barsati) on the roof. DO NOT count this as a story.  
First count functional floors. Note staircase cabins explicitly. Subtract cabin to get final count. 
A story must be a full functional floor spanning the majority of the house footprint.

You must return a valid JSON object strictly matching this schema:
{
  "Image_Context": "Exterior_Independent_House | Exterior_Apartment_Building | Interior_Flat_or_Room | Invalid_or_Unviewable", 
  "Housing_Category": "Kutcha | Semi-Pucca | Pucca | Premium | N/A",
  "Exterior_Features": { 
    "Stories": "String - e.g., 'One Storied', 'Two Storied', 'Three Storied'.",
    "Roof_Type": "Concrete | Corrugated Tin/Metal | Thatch/Tarpaulin | Asbestos | Khaprail | N/A",
    "Wall_Type": "Finished Concrete/Plaster | Exposed Brick | Mud/Makeshift | N/A",
  },
  "Interior_Features": { 
    "Floor_Material": "Tiles/Marble | Cement/Concrete | Mud/Earth | N/A", 
    "Wall_Finish": "Painted | Raw_Plaster | Bare_Brick | Mud | Tiles | N/A" 
  }, 
  "Visible_Assets_For_Income": ["List of high-value items visible (AC, Tractor, Truck, Two-Wheeler, Four-Wheeler). Empty list [] if none."], 
  "Structural_Score": "Float - A strict rating between 0.0 and 1.0. A pristine concrete house is 1.0. A high-quality/well-maintained masonry house is 0.8 to 0.9. A structurally sound brick/tin house is ~0.6 to 0.7. A house with fair to moderate deterioration is 0.4 to 0.5. A collapsing makeshift hut is 0.1 to 0.3.",
  "Structural_Condition": "Excellent | Average | Poor | Dilapidated | N/A",
}
"""

# HELPER FUNCTIONS
def analyze_image_with_gemini(image_path):
    """Sends the image and prompt to the Gemini API and parses the JSON result."""
    try:
        img = Image.open(image_path)
        response = model.generate_content([SYSTEM_PROMPT, img])
        return json.loads(response.text)
        
    except Exception as e:
        error_msg = str(e)
        print(f"Error processing {image_path}: {error_msg}")
        if "429" in error_msg or "quota" in error_msg.lower():
            raise RuntimeError("API_QUOTA_EXCEEDED")
        return None

def flatten_features(filename, features):
    """Flattens the nested JSON so each feature becomes an isolated column for ML training."""
    ext = features.get("Exterior_Features", {})
    int_feat = features.get("Interior_Features", {})
    assets = features.get("Visible_Assets_For_Income", [])
    
    # Handle edge case where assets might not be a list
    if not isinstance(assets, list):
        assets = []
        
    return {
        "Image_Name": filename,
        "Image_Context": features.get("Image_Context", "Unknown"),
        "Housing_Category": features.get("Housing_Category", "N/A"),
        
        # Flattened Exterior Features
        "Ext_Reasoning": ext.get("Reasoning", ""),
        "Ext_Stories": ext.get("Stories", "N/A"),
        "Ext_Roof_Type": ext.get("Roof_Type", "N/A"),
        "Ext_Wall_Type": ext.get("Wall_Type", "N/A"),
        
        # Flattened Interior Features
        "Int_Floor_Material": int_feat.get("Floor_Material", "N/A"),
        "Int_Wall_Finish": int_feat.get("Wall_Finish", "N/A"),
        
        # Joined Assets for easy CSV storage (e.g., "AC, Two-Wheeler")
        "Visible_Assets_For_Income": ", ".join(assets) if assets else "None",
        
        # Overall Score
        "Overall_Structural_Score": features.get("Structural_Score", None),
        "Ext_Structural_Condition": features.get("Structural_Condition", "N/A"),
        "Processing_Status": "Success"
    }

# MAIN PIPELINE
def run_folder_extraction(image_folder, output_excel_path):
    print(f"Scanning directory: {image_folder}")
    
    # Get all valid image files
    valid_extensions = ('.jpg', '.jpeg', '.png')
    all_files = [f for f in os.listdir(image_folder) if f.lower().endswith(valid_extensions)]
    
    if not all_files:
        print("No images found in the specified directory.")
        return
        
    vision_data = []
    processed_images = set()

    # Resume Logic: Check for existing output file to avoid re-processing
    if os.path.exists(output_excel_path):
        existing_df = pd.read_excel(output_excel_path)
        if 'Image_Name' in existing_df.columns:
            processed_images = set(existing_df['Image_Name'])
            print(f"Found {len(processed_images)} previously processed images. Resuming...")
            vision_data = existing_df.to_dict('records')
    
    print(f"\nFound {len(all_files)} total images. Sending to Gemini API...")
    
    try:
        for index, filename in enumerate(all_files):
            if filename in processed_images:
                continue

            image_path = os.path.join(image_folder, filename)
            print(f"[{index + 1}/{len(all_files)}] Analyzing {filename}...")
            
            raw_features = analyze_image_with_gemini(image_path)
            
            if raw_features:
                flat_row = flatten_features(filename, raw_features)
                vision_data.append(flat_row)
                
                # RATE LIMITING: Adjust sleep time based on your tier limits
                time.sleep(20) 
            else:
                vision_data.append({"Image_Name": filename, "Processing_Status": "API Error"})

    except RuntimeError as re:
        if str(re) == "API_QUOTA_EXCEEDED":
            print("\n[STOPPED] You have hit the Google Gemini Free Tier Quota.")
            print("Saving your extracted features so far...")
    except KeyboardInterrupt:
        print("\n[INTERRUPTED] Script stopped by user. Saving progress...")
    except Exception as e:
        print(f"\n[STOPPED] An unexpected error occurred: {e}")
        print("Saving your extracted features so far...")

    # Save Results to Excel
    if vision_data:
        final_df = pd.DataFrame(vision_data)
        final_df.to_excel(output_excel_path, index=False, engine='openpyxl')
        print(f"\n--- Success! Dataset saved to {output_excel_path} with {len(vision_data)} records processed ---")
    else:
        print("\nNo new data was processed.")

# EXECUTION
if __name__ == "__main__":
    if API_KEY == "YOUR_API_KEY_HERE":
        print("WAIT! You need to paste your actual Gemini API key at the top of the script first.")
    else:
        IMAGE_DIRECTORY = "./IMAGES/"
        OUTPUT_DATASET = "ML_Training_Dataset.xlsx"
        
        if not os.path.exists(IMAGE_DIRECTORY):
            os.makedirs(IMAGE_DIRECTORY)
            print(f"Created folder '{IMAGE_DIRECTORY}'. Please put your images there and re-run.")
        else:
            run_folder_extraction(IMAGE_DIRECTORY, OUTPUT_DATASET)

C:\Users\Harsh Datt\AppData\Local\Temp\ipykernel_17604\1591470689.py:6: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  import google.generativeai as genai


Scanning directory: ./IMAGES/
Found 1963 previously processed images. Resuming...

Found 2030 total images. Sending to Gemini API...
[1952/2030] Analyzing Screenshot 2026-06-28 221337.png...
[1953/2030] Analyzing Screenshot 2026-06-28 221358.png...
[1954/2030] Analyzing Screenshot 2026-06-28 221410.png...
[1955/2030] Analyzing Screenshot 2026-06-28 221426.png...
[1956/2030] Analyzing Screenshot 2026-06-28 221448.png...
[1957/2030] Analyzing Screenshot 2026-06-28 221523.png...
[1958/2030] Analyzing Screenshot 2026-06-28 221541.png...
[1959/2030] Analyzing Screenshot 2026-06-28 221558.png...
[1960/2030] Analyzing Screenshot 2026-06-28 221614.png...
[1961/2030] Analyzing Screenshot 2026-06-28 221629.png...
[1962/2030] Analyzing Screenshot 2026-06-28 221644.png...
[1963/2030] Analyzing Screenshot 2026-06-28 221755.png...
[1964/2030] Analyzing Screenshot 2026-06-28 221810.png...
[1965/2030] Analyzing Screenshot 2026-06-28 221819.png...
[1966/2030] Analyzing Screenshot 2026-06-28 221844.png.

In [ ]:
import os
import json
import time
import pandas as pd
from PIL import Image
import google.generativeai as genai

# SETUP & CONFIGURATION
API_KEY = "xxxxxxxxxxxxxxxxxxxx" # Replace with your actual Gemini API key
genai.configure(api_key=API_KEY)

# Using gemini-2.5-flash for rapid, cost-effective vision processing
model = genai.GenerativeModel(
    model_name="gemini-2.5-flash",
    generation_config={"response_mime_type": "application/json"}
)

# PROMPT ENGINEERING
SYSTEM_PROMPT = """
You are an expert socio-economic surveyor assessing homes. 
Look closely at the provided photo and extract its structural and interior features. 

CRITICAL RULE FOR COUNTING STORIES:
Indian houses often have a small staircase head-room (mummy/barsati) on the roof. DO NOT count this as a story.  
First count functional floors. Note staircase cabins explicitly. Subtract cabin to get final count. 
A story must be a full functional floor spanning the majority of the house footprint.

You must return a valid JSON object strictly matching this schema:
{
  "Image_Context": "Exterior_Independent_House | Exterior_Apartment_Building | Interior_Flat_or_Room | Invalid_or_Unviewable", 
  "Housing_Category": "Kutcha | Semi-Pucca | Pucca | Premium | N/A",
  "Exterior_Features": { 
    "Stories": "String - e.g., 'One Storied', 'Two Storied', 'Three Storied'.",
    "Roof_Type": "Concrete | Corrugated Tin/Metal | Thatch/Tarpaulin | Asbestos | Khaprail | N/A",
    "Wall_Type": "Finished Concrete/Plaster | Exposed Brick | Mud/Makeshift | N/A",
  },
  "Interior_Features": { 
    "Floor_Material": "Tiles/Marble | Cement/Concrete | Mud/Earth | N/A", 
    "Wall_Finish": "Painted | Raw_Plaster | Bare_Brick | Mud | Tiles | N/A" 
  }, 
  "Visible_Assets_For_Income": ["List of high-value items visible (AC, Tractor, Truck, Two-Wheeler, Four-Wheeler). Empty list [] if none."], 
  "Structural_Score": "Float - A strict rating between 0.0 and 1.0. A pristine concrete house is 1.0. A high-quality/well-maintained masonry house is 0.8 to 0.9. A structurally sound brick/tin house is ~0.6 to 0.7. A house with fair to moderate deterioration is 0.4 to 0.5. A collapsing makeshift hut is 0.1 to 0.3.",
  "Structural_Condition": "Excellent | Average | Poor | Dilapidated | N/A",
}
"""

# HELPER FUNCTIONS
def analyze_image_with_gemini(image_path):
    """Sends the image and prompt to the Gemini API and parses the JSON result."""
    try:
        img = Image.open(image_path)
        response = model.generate_content([SYSTEM_PROMPT, img])
        return json.loads(response.text)
        
    except Exception as e:
        error_msg = str(e)
        print(f"Error processing {image_path}: {error_msg}")
        if "429" in error_msg or "quota" in error_msg.lower():
            raise RuntimeError("API_QUOTA_EXCEEDED")
        return None

def flatten_features(filename, features):
    """Flattens the nested JSON so each feature becomes an isolated column for ML training."""
    ext = features.get("Exterior_Features", {})
    int_feat = features.get("Interior_Features", {})
    assets = features.get("Visible_Assets_For_Income", [])
    
    # Handle edge case where assets might not be a list
    if not isinstance(assets, list):
        assets = []
        
    return {
        "Image_Name": filename,
        "Image_Context": features.get("Image_Context", "Unknown"),
        "Housing_Category": features.get("Housing_Category", "N/A"),
        
        # Flattened Exterior Features
        "Ext_Reasoning": ext.get("Reasoning", ""),
        "Ext_Stories": ext.get("Stories", "N/A"),
        "Ext_Roof_Type": ext.get("Roof_Type", "N/A"),
        "Ext_Wall_Type": ext.get("Wall_Type", "N/A"),
        
        # Flattened Interior Features
        "Int_Floor_Material": int_feat.get("Floor_Material", "N/A"),
        "Int_Wall_Finish": int_feat.get("Wall_Finish", "N/A"),
        
        # Joined Assets for easy CSV storage (e.g., "AC, Two-Wheeler")
        "Visible_Assets_For_Income": ", ".join(assets) if assets else "None",
        
        # Overall Score
        "Overall_Structural_Score": features.get("Structural_Score", None),
        "Ext_Structural_Condition": features.get("Structural_Condition", "N/A"),
        "Processing_Status": "Success"
    }

# MAIN PIPELINE
def run_folder_extraction(image_folder, output_excel_path):
    print(f"Scanning directory: {image_folder}")
    
    # Get all valid image files
    valid_extensions = ('.jpg', '.jpeg', '.png')
    all_files = [f for f in os.listdir(image_folder) if f.lower().endswith(valid_extensions)]
    
    if not all_files:
        print("No images found in the specified directory.")
        return
        
    vision_data = []
    processed_images = set()

    # Resume Logic: Check for existing output file to avoid re-processing
    if os.path.exists(output_excel_path):
        existing_df = pd.read_excel(output_excel_path)
        if 'Image_Name' in existing_df.columns:
            processed_images = set(existing_df['Image_Name'])
            print(f"Found {len(processed_images)} previously processed images. Resuming...")
            vision_data = existing_df.to_dict('records')
    
    print(f"\nFound {len(all_files)} total images. Sending to Gemini API...")
    
    try:
        for index, filename in enumerate(all_files):
            if filename in processed_images:
                continue

            image_path = os.path.join(image_folder, filename)
            print(f"[{index + 1}/{len(all_files)}] Analyzing {filename}...")
            
            raw_features = analyze_image_with_gemini(image_path)
            
            if raw_features:
                flat_row = flatten_features(filename, raw_features)
                vision_data.append(flat_row)
                
                # RATE LIMITING: Adjust sleep time based on your tier limits
                time.sleep(20) 
            else:
                vision_data.append({"Image_Name": filename, "Processing_Status": "API Error"})

    except RuntimeError as re:
        if str(re) == "API_QUOTA_EXCEEDED":
            print("\n[STOPPED] You have hit the Google Gemini Free Tier Quota.")
            print("Saving your extracted features so far...")
    except KeyboardInterrupt:
        print("\n[INTERRUPTED] Script stopped by user. Saving progress...")
    except Exception as e:
        print(f"\n[STOPPED] An unexpected error occurred: {e}")
        print("Saving your extracted features so far...")

    # Save Results to Excel
    if vision_data:
        final_df = pd.DataFrame(vision_data)
        final_df.to_excel(output_excel_path, index=False, engine='openpyxl')
        print(f"\n--- Success! Dataset saved to {output_excel_path} with {len(vision_data)} records processed ---")
    else:
        print("\nNo new data was processed.")

# EXECUTION
if __name__ == "__main__":
    if API_KEY == "YOUR_API_KEY_HERE":
        print("WAIT! You need to paste your actual Gemini API key at the top of the script first.")
    else:
        IMAGE_DIRECTORY = "./IMAGES/"
        OUTPUT_DATASET = "ML_Training_Dataset.xlsx"
        
        if not os.path.exists(IMAGE_DIRECTORY):
            os.makedirs(IMAGE_DIRECTORY)
            print(f"Created folder '{IMAGE_DIRECTORY}'. Please put your images there and re-run.")
        else:
            run_folder_extraction(IMAGE_DIRECTORY, OUTPUT_DATASET)

Scanning directory: ./IMAGES/
Found 1984 previously processed images. Resuming...

Found 2030 total images. Sending to Gemini API...
[1973/2030] Analyzing Screenshot 2026-06-28 222024.png...
[1974/2030] Analyzing Screenshot 2026-06-28 222050.png...
[1975/2030] Analyzing Screenshot 2026-06-28 222115.png...
[1976/2030] Analyzing Screenshot 2026-06-28 222127.png...
[1977/2030] Analyzing Screenshot 2026-06-28 222141.png...
[1978/2030] Analyzing Screenshot 2026-06-28 222156.png...
[1979/2030] Analyzing Screenshot 2026-06-28 222211.png...
[1980/2030] Analyzing Screenshot 2026-06-28 222223.png...
Error processing ./IMAGES/Screenshot 2026-06-28 222223.png: 504 Stream removed (Deadline Exceeded)
[1981/2030] Analyzing Screenshot 2026-06-28 222237.png...
Error processing ./IMAGES/Screenshot 2026-06-28 222237.png: 504 Deadline Exceeded
[1982/2030] Analyzing Screenshot 2026-06-28 222252.png...
Error processing ./IMAGES/Screenshot 2026-06-28 222252.png: 504 Stream removed (Deadline Exceeded)
[1983/2

KeyboardInterrupt: 

In [3]:
import sys
import pandas as pd

# 1. Grab the last active traceback frame from the crash
tb = getattr(sys, "last_traceback", None)

if tb is None:
    print("❌ Could not find the last crash frame. Did you restart the kernel?")
else:
    vision_data = None
    final_df = None
    
    # 2. Walk through the crash frames to find your function's local variables
    while tb:
        frame_locals = tb.tb_frame.f_locals
        
        # Look for the variables inside your run_folder_extraction loop
        if 'vision_data' in frame_locals:
            vision_data = frame_locals['vision_data']
        if 'final_df' in frame_locals:
            final_df = frame_locals['final_df']
            
        tb = tb.tb_next

    # 3. Save the data safely!
    if final_df is not None:
        print(f"🎉 SUCCESS! Found the fully built DataFrame in memory ({len(final_df)} records).")
        final_df.to_excel("EMERGENCY_RESCUED_DATA.xlsx", index=False, engine='openpyxl')
        print("💾 Saved safely to 'EMERGENCY_RESCUED_DATA.xlsx'!")
        
    elif vision_data is not None:
        print(f"🎉 SUCCESS! Found the raw list of records in memory ({len(vision_data)} items).")
        rescued_df = pd.DataFrame(vision_data)
        rescued_df.to_excel("EMERGENCY_RESCUED_DATA.xlsx", index=False, engine='openpyxl')
        print("💾 Converted and saved safely to 'EMERGENCY_RESCUED_DATA.xlsx'!")
        
    else:
        print("❌ Could not find 'vision_data' or 'final_df' in the local variables of the crash stack.")

🎉 SUCCESS! Found the fully built DataFrame in memory (1994 records).
💾 Saved safely to 'EMERGENCY_RESCUED_DATA.xlsx'!
